In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Now loading the canonical dataset

import pandas as pd
import sqlite3
import os

analysis_file_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/processed/analysis_ready_v2.csv"
)

df = pd.read_csv(
    analysis_file_path,
    encoding="utf-8-sig",
    dtype={"lor": "string"},
    parse_dates=[
        "record_created_date",
        "offence_start_date",
        "offence_end_date",
        "offence_start",
        "offence_end"
    ]
)

print("Rows and columns:", df.shape)
print("LOR type:", df["lor"].dtype)
print(
    "All LOR codes have 8 characters:",
    df["lor"].str.len().eq(8).all()
)

Rows and columns: (26965, 34)
LOR type: string
All LOR codes have 8 characters: True


In [3]:
# Creating the SQLite database locally in Colab

database_path = "/content/berlin_bicycle_theft_v1.db"

connection = sqlite3.connect(database_path)

df.to_sql(
    "bike_theft",
    connection,
    if_exists="replace",
    index=False
)

database_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS row_count,
        MIN(LENGTH(lor)) AS minimum_lor_length,
        MAX(LENGTH(lor)) AS maximum_lor_length
    FROM bike_theft;
    """,
    connection
)

display(database_check)

,row_count,minimum_lor_length,maximum_lor_length
0,26965,8,8


In [4]:
# Creating the first SQL query: a dataset overview

overview_query = """
SELECT
    COUNT(*) AS total_reports,
    MIN(DATE(offence_start_date)) AS earliest_date,
    MAX(DATE(offence_start_date)) AS latest_date,
    COUNT(DISTINCT lor) AS planning_areas_represented,
    COUNT(DISTINCT district_id) AS districts_represented
FROM bike_theft;
"""

overview_result = pd.read_sql_query(
    overview_query,
    connection
)

display(overview_result)

,total_reports,earliest_date,latest_date,planning_areas_represented,districts_represented
0,26965,2025-01-01,2026-08-08,534,12


In [5]:
# Reproducing the monthly report counts with SQL

monthly_query = """
SELECT
    STRFTIME('%Y-%m', offence_start_date) AS year_month,
    COUNT(*) AS reported_incidents
FROM bike_theft
GROUP BY
    STRFTIME('%Y-%m', offence_start_date)
ORDER BY
    year_month;
"""

monthly_sql = pd.read_sql_query(
    monthly_query,
    connection
)

display(monthly_sql)

print(
    "Total reports:",
    monthly_sql["reported_incidents"].sum()
)

,year_month,reported_incidents
0,2025-01,1155
1,2025-02,947
2,2025-03,1547
3,2025-04,1603
4,2025-05,1948
5,2025-06,1868
6,2025-07,1797
7,2025-08,1353
8,2025-09,1545
9,2025-10,1346


Total reports: 26965


In [6]:
# creating the fair 2025–2026 comparison using CTEs and a window function

ytd_query = """
WITH comparable_reports AS (
    SELECT
        CAST(
            STRFTIME('%Y', offence_start_date)
            AS INTEGER
        ) AS year
    FROM bike_theft
    WHERE
        DATE(offence_start_date)
            BETWEEN '2025-01-01' AND '2025-08-08'
        OR
        DATE(offence_start_date)
            BETWEEN '2026-01-01' AND '2026-08-08'
),

year_counts AS (
    SELECT
        year,
        COUNT(*) AS reported_incidents
    FROM comparable_reports
    GROUP BY year
),

year_comparison AS (
    SELECT
        year,
        reported_incidents,
        LAG(reported_incidents) OVER (
            ORDER BY year
        ) AS previous_year_reports
    FROM year_counts
)

SELECT
    year,
    reported_incidents,
    reported_incidents
        - previous_year_reports AS difference,
    ROUND(
        100.0
        * (reported_incidents - previous_year_reports)
        / previous_year_reports,
        1
    ) AS percentage_change
FROM year_comparison
ORDER BY year;
"""

ytd_sql = pd.read_sql_query(
    ytd_query,
    connection
)

display(ytd_sql)

,year,reported_incidents,difference,percentage_change
0,2025,11185,NaN,NaN
1,2026,9629,-1556.0,-13.9


In [7]:
# Reproducing the weekday analysis using SQL

weekday_query = """
WITH daily_counts AS (
    SELECT
        DATE(offence_start_date) AS offence_date,
        CAST(
            STRFTIME('%w', offence_start_date)
            AS INTEGER
        ) AS weekday_number,
        COUNT(*) AS reported_incidents
    FROM bike_theft
    GROUP BY
        DATE(offence_start_date)
)

SELECT
    CASE weekday_number
        WHEN 0 THEN 'Sunday'
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
    END AS weekday_name,

    COUNT(*) AS calendar_days,
    SUM(reported_incidents) AS total_reports,
    ROUND(
        AVG(reported_incidents),
        1
    ) AS average_reports_per_day

FROM daily_counts

GROUP BY
    weekday_number

ORDER BY
    CASE weekday_number
        WHEN 1 THEN 1
        WHEN 2 THEN 2
        WHEN 3 THEN 3
        WHEN 4 THEN 4
        WHEN 5 THEN 5
        WHEN 6 THEN 6
        WHEN 0 THEN 7
    END;
"""

weekday_sql = pd.read_sql_query(
    weekday_query,
    connection
)

display(weekday_sql)

,weekday_name,calendar_days,total_reports,average_reports_per_day
0,Monday,83,4018,48.4
1,Tuesday,83,4041,48.7
2,Wednesday,84,4113,49.0
3,Thursday,84,4020,47.9
4,Friday,84,4211,50.1
5,Saturday,84,3611,43.0
6,Sunday,83,2951,35.6


In [8]:
# Next ranking bicycle types and calculating their percentage of completed incidents using SQL window functions

bicycle_type_query = """
WITH completed_reports AS (
    SELECT
        CASE bicycle_type
            WHEN 'Herrenfahrrad'
                THEN 'Men''s bicycle'
            WHEN 'Damenfahrrad'
                THEN 'Women''s bicycle'
            WHEN 'Fahrrad'
                THEN 'Bicycle (unspecified)'
            WHEN 'Mountainbike'
                THEN 'Mountain bike'
            WHEN 'Kinderfahrrad'
                THEN 'Children''s bicycle'
            WHEN 'Rennrad'
                THEN 'Road bicycle'
            WHEN 'diverse Fahrräder'
                THEN 'Various bicycles'
            WHEN 'Trekkingrad'
                THEN 'Trekking bicycle'
            WHEN 'Citybike'
                THEN 'City bicycle'
            WHEN 'Lastenfahrrad'
                THEN 'Cargo bicycle'
            WHEN 'Hollandrad'
                THEN 'Dutch-style bicycle'
            WHEN 'Klapprad'
                THEN 'Folding bicycle'
            WHEN 'BMX'
                THEN 'BMX'
            ELSE bicycle_type
        END AS bicycle_type
    FROM bike_theft
    WHERE attempt = 'Nein'
),

type_counts AS (
    SELECT
        bicycle_type,
        COUNT(*) AS reported_incidents
    FROM completed_reports
    GROUP BY bicycle_type
)

SELECT
    DENSE_RANK() OVER (
        ORDER BY reported_incidents DESC
    ) AS type_rank,
    bicycle_type,
    reported_incidents,
    ROUND(
        100.0 * reported_incidents
        / SUM(reported_incidents) OVER (),
        1
    ) AS share_pct
FROM type_counts
ORDER BY type_rank;
"""

bicycle_type_sql = pd.read_sql_query(
    bicycle_type_query,
    connection
)

display(bicycle_type_sql)

,type_rank,bicycle_type,reported_incidents,share_pct
0,1,Men's bicycle,10877,40.6
1,2,Women's bicycle,5902,22.0
2,3,Bicycle (unspecified),4676,17.4
3,4,Mountain bike,1308,4.9
4,5,Children's bicycle,1201,4.5
5,6,Road bicycle,909,3.4
6,7,Various bicycles,651,2.4
7,8,Trekking bicycle,514,1.9
8,9,City bicycle,503,1.9
9,10,Cargo bicycle,133,0.5


In [9]:
# Reproducing the financial analysis by bicycle type (SQLite has no built-in MEDIAN(), so this query calculates the median using row numbers and window functions)

financial_type_query = """
WITH eligible_reports AS (
    SELECT
        CASE bicycle_type
            WHEN 'Herrenfahrrad'
                THEN 'Men''s bicycle'
            WHEN 'Damenfahrrad'
                THEN 'Women''s bicycle'
            WHEN 'Fahrrad'
                THEN 'Bicycle (unspecified)'
            WHEN 'Mountainbike'
                THEN 'Mountain bike'
            WHEN 'Kinderfahrrad'
                THEN 'Children''s bicycle'
            WHEN 'Rennrad'
                THEN 'Road bicycle'
            WHEN 'diverse Fahrräder'
                THEN 'Various bicycles'
            WHEN 'Trekkingrad'
                THEN 'Trekking bicycle'
            WHEN 'Citybike'
                THEN 'City bicycle'
            WHEN 'Lastenfahrrad'
                THEN 'Cargo bicycle'
            WHEN 'Hollandrad'
                THEN 'Dutch-style bicycle'
            WHEN 'Klapprad'
                THEN 'Folding bicycle'
            WHEN 'BMX'
                THEN 'BMX'
            ELSE bicycle_type
        END AS bicycle_type,

        reported_damage_eur

    FROM bike_theft

    WHERE
        attempt = 'Nein'
        AND is_cellar_burglary = 0
        AND reported_damage_eur > 0
),

ordered_values AS (
    SELECT
        bicycle_type,
        reported_damage_eur,

        ROW_NUMBER() OVER (
            PARTITION BY bicycle_type
            ORDER BY reported_damage_eur
        ) AS value_position,

        COUNT(*) OVER (
            PARTITION BY bicycle_type
        ) AS record_count

    FROM eligible_reports
),

type_summary AS (
    SELECT
        bicycle_type,
        COUNT(*) AS reports,

        ROUND(
            AVG(
                CASE
                    WHEN value_position IN (
                        (record_count + 1) / 2,
                        (record_count + 2) / 2
                    )
                    THEN reported_damage_eur
                END
            ),
            1
        ) AS median_damage_eur,

        ROUND(
            AVG(reported_damage_eur),
            1
        ) AS mean_damage_eur,

        SUM(reported_damage_eur)
            AS total_damage_eur

    FROM ordered_values

    GROUP BY bicycle_type
    HAVING COUNT(*) >= 20
)

SELECT
    DENSE_RANK() OVER (
        ORDER BY median_damage_eur DESC
    ) AS median_rank,

    bicycle_type,
    reports,
    median_damage_eur,
    mean_damage_eur,
    total_damage_eur

FROM type_summary
ORDER BY median_rank;
"""

financial_type_sql = pd.read_sql_query(
    financial_type_query,
    connection
)

display(financial_type_sql)

,median_rank,bicycle_type,reports,median_damage_eur,mean_damage_eur,total_damage_eur
0,1,Cargo bicycle,127,3000.0,3445.6,437597
1,2,Road bicycle,775,1273.0,1471.5,1140425
2,3,Various bicycles,467,1200.0,2031.0,948479
3,4,Bicycle (unspecified),4467,1049.0,1416.9,6329253
4,5,Men's bicycle,10443,1000.0,1341.0,14004284
5,6,City bicycle,476,949.5,1312.5,624756
6,7,Trekking bicycle,490,940.0,1265.1,619881
7,8,Women's bicycle,5709,728.0,1047.1,5978169
8,9,Folding bicycle,50,649.5,962.3,48114
9,10,Mountain bike,1143,625.0,1053.0,1203596


In [10]:
# Demonstrating a proper SQL JOIN -> creating a separate LOR lookup table from the canonical data, then joining it back to the incident records to calculate district totals

connection.executescript(
    """
    DROP TABLE IF EXISTS lor_lookup;

    CREATE TABLE lor_lookup AS
    SELECT DISTINCT
        lor,
        district_id,
        district_name,
        forecast_area_id,
        forecast_area_name,
        district_region_id,
        district_region_name,
        planning_area_name
    FROM bike_theft;

    CREATE UNIQUE INDEX
        idx_lor_lookup_code
    ON lor_lookup(lor);
    """
)

district_join_query = """
SELECT
    lookup.district_name,
    COUNT(*) AS reported_incidents,
    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct,

    DENSE_RANK() OVER (
        ORDER BY COUNT(*) DESC
    ) AS report_rank

FROM bike_theft AS incidents

INNER JOIN lor_lookup AS lookup
    ON incidents.lor = lookup.lor

GROUP BY
    lookup.district_name

ORDER BY
    report_rank;
"""

district_sql = pd.read_sql_query(
    district_join_query,
    connection
)

display(district_sql)

print(
    "LOR lookup rows:",
    pd.read_sql_query(
        "SELECT COUNT(*) AS count FROM lor_lookup;",
        connection
    )["count"].iloc[0]
)

print(
    "Reports preserved after join:",
    district_sql["reported_incidents"].sum()
)

,district_name,reported_incidents,share_pct,report_rank
0,Mitte,5130,19.0,1
1,Friedrichshain-Kreuzberg,4001,14.8,2
2,Pankow,3193,11.8,3
3,Charlottenburg-Wilmersdorf,2476,9.2,4
4,Tempelhof-Schöneberg,2469,9.2,5
5,Neukölln,2041,7.6,6
6,Lichtenberg,1795,6.7,7
7,Treptow-Köpenick,1758,6.5,8
8,Steglitz-Zehlendorf,1628,6.0,9
9,Reinickendorf,1037,3.8,10


LOR lookup rows: 534
Reports preserved after join: 26965


In [11]:
# Now saving all queries as one .sql file in 03_SQL

sql_folder = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "03_SQL"
)

sql_file_path = (
    sql_folder
    + "/P1_core_queries_v1.sql"
)

os.makedirs(sql_folder, exist_ok=True)

lookup_setup_sql = """
DROP TABLE IF EXISTS lor_lookup;

CREATE TABLE lor_lookup AS
SELECT DISTINCT
    lor,
    district_id,
    district_name,
    forecast_area_id,
    forecast_area_name,
    district_region_id,
    district_region_name,
    planning_area_name
FROM bike_theft;

CREATE UNIQUE INDEX
    idx_lor_lookup_code
ON lor_lookup(lor);
"""

sql_script_content = f"""
-- Dude, Where's My Bike?
-- Core SQL analysis
-- Canonical source: analysis_ready_v2.csv
-- Table: bike_theft

-- 1. Dataset overview
{overview_query.strip()}

-- 2. Monthly report counts
{monthly_query.strip()}

-- 3. Matching 2025–2026 YTD comparison
{ytd_query.strip()}

-- 4. Weekday analysis
{weekday_query.strip()}

-- 5. Bicycle-type ranking
{bicycle_type_query.strip()}

-- 6. Financial analysis by bicycle type
{financial_type_query.strip()}

-- 7. LOR lookup table
{lookup_setup_sql.strip()}

-- 8. District ranking through an SQL join
{district_join_query.strip()}
"""

if os.path.exists(sql_file_path):
    print("SQL file already exists. Nothing was overwritten:")
    print(sql_file_path)

else:
    with open(
        sql_file_path,
        "w",
        encoding="utf-8"
    ) as sql_file:
        sql_file.write(sql_script_content)

    os.sync()

    print("SQL script saved:")
    print(sql_file_path)
    print(
        "File size:",
        os.path.getsize(sql_file_path),
        "bytes"
    )

SQL script saved:
/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/03_SQL/P1_core_queries_v1.sql
File size: 7439 bytes


In [12]:
# Now backing up the SQLite database to 03_SQL

import shutil

# Finish pending database writes
connection.commit()
connection.close()

drive_database_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "03_SQL/berlin_bicycle_theft_v1.db"
)

if os.path.exists(drive_database_path):
    print("Database already exists. Nothing was overwritten:")
    print(drive_database_path)

else:
    shutil.copy2(
        database_path,
        drive_database_path
    )

    os.sync()

    print("Database saved:")
    print(drive_database_path)

# Verify the saved database
verification_connection = sqlite3.connect(
    drive_database_path
)

saved_database_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS report_rows
    FROM bike_theft;
    """,
    verification_connection
)

lookup_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS lookup_rows
    FROM lor_lookup;
    """,
    verification_connection
)

verification_connection.close()

display(saved_database_check)
display(lookup_check)

Database saved:
/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/03_SQL/berlin_bicycle_theft_v1.db


,report_rows
0,26965


,lookup_rows
0,534


Testing the rounding hypothesis:

Are reported damage values strongly rounded?
If many amounts are exactly €500, €1,000 or €1,500, that would support describing them as estimates rather than precise bicycle prices.

In [15]:
connection = sqlite3.connect(database_path)

In [14]:
rounding_query = """
WITH eligible_values AS (
    SELECT
        reported_damage_eur
    FROM bike_theft
    WHERE
        attempt = 'Nein'
        AND is_cellar_burglary = 0
        AND reported_damage_eur > 0
)

SELECT
    COUNT(*) AS positive_value_records,

    SUM(
        CASE
            WHEN reported_damage_eur IN (500, 1000, 1500)
            THEN 1
            ELSE 0
        END
    ) AS records_at_500_1000_1500,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN reported_damage_eur IN (500, 1000, 1500)
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS selected_round_values_pct,

    SUM(
        CASE
            WHEN reported_damage_eur % 100 = 0
            THEN 1
            ELSE 0
        END
    ) AS multiples_of_100,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN reported_damage_eur % 100 = 0
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS multiples_of_100_pct,

    SUM(
        CASE
            WHEN reported_damage_eur % 500 = 0
            THEN 1
            ELSE 0
        END
    ) AS multiples_of_500,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN reported_damage_eur % 500 = 0
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS multiples_of_500_pct

FROM eligible_values;
"""

rounding_summary = pd.read_sql_query(
    rounding_query,
    connection
)

display(rounding_summary)

,positive_value_records,records_at_500_1000_1500,selected_round_values_pct,multiples_of_100,multiples_of_100_pct,multiples_of_500,multiples_of_500_pct
0,25386,1849,7.3,8608,33.9,3041,12.0


In [16]:
# identifying the most common exact values:

common_values_query = """
SELECT
    reported_damage_eur,
    COUNT(*) AS records,
    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct
FROM bike_theft
WHERE
    attempt = 'Nein'
    AND is_cellar_burglary = 0
    AND reported_damage_eur > 0
GROUP BY
    reported_damage_eur
ORDER BY
    records DESC
LIMIT 15;
"""

common_values = pd.read_sql_query(
    common_values_query,
    connection
)

display(common_values)

,reported_damage_eur,records,share_pct
0,500,807,3.2
1,1000,615,2.4
2,300,552,2.2
3,400,504,2.0
4,600,497,2.0
5,2000,439,1.7
6,1500,427,1.7
7,1200,416,1.6
8,700,392,1.5
9,800,390,1.5


Reported damage values show substantial rounding. One-third of positive values are exact multiples of €100, and the most common recorded amounts are round figures such as €500, €1,000 and €300. This suggests that many values are reported estimates rather than precise verified bicycle prices.

In [17]:
# Then testing weekend patterns by district:

district_weekend_query = """
WITH RECURSIVE date_bounds AS (
    SELECT
        MIN(DATE(offence_start_date)) AS first_date,
        MAX(DATE(offence_start_date)) AS last_date
    FROM bike_theft
),

calendar(calendar_date) AS (
    SELECT first_date
    FROM date_bounds

    UNION ALL

    SELECT DATE(calendar_date, '+1 day')
    FROM calendar, date_bounds
    WHERE calendar_date < last_date
),

calendar_totals AS (
    SELECT
        SUM(
            CASE
                WHEN STRFTIME('%w', calendar_date)
                    IN ('0', '6')
                THEN 1
                ELSE 0
            END
        ) AS weekend_days,

        SUM(
            CASE
                WHEN STRFTIME('%w', calendar_date)
                    NOT IN ('0', '6')
                THEN 1
                ELSE 0
            END
        ) AS weekday_days
    FROM calendar
),

district_counts AS (
    SELECT
        district_name,

        SUM(
            CASE
                WHEN STRFTIME('%w', offence_start_date)
                    IN ('0', '6')
                THEN 1
                ELSE 0
            END
        ) AS weekend_reports,

        SUM(
            CASE
                WHEN STRFTIME('%w', offence_start_date)
                    NOT IN ('0', '6')
                THEN 1
                ELSE 0
            END
        ) AS weekday_reports

    FROM bike_theft
    GROUP BY district_name
)

SELECT
    district_name,
    weekday_reports,
    weekend_reports,

    ROUND(
        1.0 * weekday_reports / weekday_days,
        1
    ) AS weekday_average,

    ROUND(
        1.0 * weekend_reports / weekend_days,
        1
    ) AS weekend_average,

    ROUND(
        100.0 * (
            1.0 * weekend_reports / weekend_days
            - 1.0 * weekday_reports / weekday_days
        )
        / (1.0 * weekday_reports / weekday_days),
        1
    ) AS weekend_difference_pct

FROM district_counts
CROSS JOIN calendar_totals

ORDER BY weekend_difference_pct DESC;
"""

district_weekend_sql = pd.read_sql_query(
    district_weekend_query,
    connection
)

display(district_weekend_sql)

,district_name,weekday_reports,weekend_reports,weekday_average,weekend_average,weekend_difference_pct
0,Neukölln,1493,548,3.6,3.3,-8.1
1,Spandau,510,182,1.2,1.1,-10.7
2,Charlottenburg-Wilmersdorf,1841,635,4.4,3.8,-13.7
3,Friedrichshain-Kreuzberg,2988,1013,7.1,6.1,-15.1
4,Marzahn-Hellersdorf,563,182,1.3,1.1,-19.1
5,Lichtenberg,1359,436,3.3,2.6,-19.7
6,Treptow-Köpenick,1333,425,3.2,2.5,-20.2
7,Tempelhof-Schöneberg,1873,596,4.5,3.6,-20.4
8,Mitte,3916,1214,9.4,7.3,-22.4
9,Pankow,2447,746,5.9,4.5,-23.7


### Additional exploratory findings

#### Rounded reported-damage values

Reported financial damage shows substantial value heaping:

- 33.9% of positive values are exact multiples of €100.
- 12.0% are multiples of €500.
- The most common individual amounts include €500, €1,000, €300 and €400.

This suggests that many recorded values are rounded estimates rather than precise verified bicycle prices. It does not mean that the values are incorrect, but financial findings should avoid false precision.

#### Weekend patterns by district

Every district recorded fewer reports per calendar day on weekends, but the size of the decline varied considerably.

- Neukölln had the smallest decline at 8.1%.
- Friedrichshain-Kreuzberg declined by 15.1%.
- Mitte declined by 22.4%.
- Steglitz-Zehlendorf had the largest decline at 31.0%.

The results do not support a simple central-versus-outer-district explanation. They show district-specific differences, but the dataset cannot explain their causes. Differences could relate to bicycle activity, commuting, land use, reporting behaviour or other factors not contained in the data.

In [18]:
supplementary_sql_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "03_SQL/P1_supplementary_queries_v1.sql"
)

supplementary_sql_content = f"""
-- Dude, Where's My Bike?
-- Supplementary exploratory SQL queries
-- Canonical source: analysis_ready_v2.csv

-- 1. Summary of rounded reported-damage values
{rounding_query.strip()}

-- 2. Most common exact reported-damage values
{common_values_query.strip()}

-- 3. Weekend versus weekday patterns by district
{district_weekend_query.strip()}
"""

if os.path.exists(supplementary_sql_path):
    print("File already exists. Nothing was overwritten:")
    print(supplementary_sql_path)

else:
    with open(
        supplementary_sql_path,
        "w",
        encoding="utf-8"
    ) as sql_file:
        sql_file.write(supplementary_sql_content)

    os.sync()

    print("Supplementary SQL file saved:")
    print(supplementary_sql_path)
    print(
        "File size:",
        os.path.getsize(supplementary_sql_path),
        "bytes"
    )

Supplementary SQL file saved:
/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/03_SQL/P1_supplementary_queries_v1.sql
File size: 3954 bytes


In [19]:
connection.close()

print("SQL work completed.")

SQL work completed.
